# Validate Florence-2 + Groq Pipeline
This notebook tests the spacing corrections and token-overlap validations.

In [ ]:
import os
import subprocess

# Download the repository to get the samples/ folder and app.py
if not os.path.exists('samples'):
    print('Samples not found. Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/Aaryan658/iam-handwriting-explainer.git', 'repo'])
    os.rename('repo/samples', 'samples')
    os.rename('repo/app.py', 'app.py')
    print('Setup complete!')
else:
    print('Samples directory already exists.')

In [ ]:
!pip install -q transformers torch pillow groq wordninja jiwer

In [ ]:
import os
import jiwer
from PIL import Image
import app

In [ ]:
SAMPLES = [
    "samples/line_01.png",
    "samples/line_02.png",
    "samples/line_03.png",
    "samples/line_05.png",
    "samples/education_paragraph.png"
]

# IMPORTANT: Set your Groq API Key here if running in Colab
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'

for sample_path in SAMPLES:
    if not os.path.exists(sample_path):
        print(f"Sample not found: {sample_path}")
        continue
        
    print(f"\n--- Validating: {sample_path} ---")
    img = Image.open(sample_path).convert("RGB")
    txt_path = sample_path.replace(".png", ".txt")
    ground_truth = ""
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            ground_truth = f.read().strip()
            
    print("Running Florence-2...")
    raw_ocr = app.transcribe(img)
    print(f"[RAW OCR]: {raw_ocr}")
    
    corrected_ocr = app.correct_spaces(raw_ocr)
    print(f"[SPACE CORRECTED]: {corrected_ocr}")
    
    if ground_truth:
        print(f"[GROUND TRUTH]: {ground_truth}")
        try:
            error = jiwer.wer(ground_truth.lower(), corrected_ocr.lower())
            accuracy = max(0.0, 1.0 - error)
            print(f"[WORD-LEVEL ACCURACY]: {accuracy:.2%}")
        except Exception as e:
            print(f"[WER CALCULATION ERROR]: {e}")
    else:
        print("[GROUND TRUTH]: None provided for this sample.")
        
    print("Running Groq explanation + Token overlap check...")
    explanation_result = app.explain(raw_ocr)
    print("\n[GROQ RESULT]:\n" + explanation_result)